# Seeds 2 and 3 --- error bars on the matched sweep

Seed 1 is complete: 100 cells, 0 failures, **19.6 GPU-hours**. These two seeds repeat it exactly, so every reported number can carry a mean and a sample sd instead of being a single draw. Budget is therefore **~39 GPU-hours, ~42 h wall clock**.

## Two ordering decisions, both learned from seed 1

**Arms interleave.** Seed 1 ran all 48 I.P. cells before any BaCP cell, so no delta existed until roughly halfway through a 21-hour run. Here each `(pruner, sparsity, model)` runs I.P. immediately followed by BaCP, so complete pairs land together and any partial state is still a usable table.

**Magnitude first, then SNIP-it, then WANDA.** If this is cut short, the criterion the paper leads with is the one that finished.

Seed-major: seed 2 completes before seed 3 starts, so an interrupted run yields n=2 everywhere rather than n=3 on a fragment.

## Sequential, deliberately

`run_parallel` was measured on this workload and is **slower**: four VGG-11 BaCP cells gave a 4.6x per-cell slowdown against an ideal 4.0x, so 0.87x aggregate throughput. A single cell already saturates the A100. See the `run_parallel` docstring in `nb_common.py`.

Safe to interrupt and re-run: a recorded cell is skipped.

In [ ]:
import sys, pathlib
here = pathlib.Path.cwd()
for cand in [here, *here.parents]:
    if (cand / 'nb_common.py').exists():
        sys.path.insert(0, str(cand)); break
    if (cand / 'project' / 'test_notebooks' / 'nb_common.py').exists():
        sys.path.insert(0, str(cand / 'project' / 'test_notebooks')); break
else:
    raise RuntimeError('cannot find nb_common.py -- start the kernel inside the repo')
import nb_common as nb
info = nb.setup()

## Sanity check

In [ ]:
SEEDS      = (2, 3)
MODELS     = ['resnet34', 'resnet50', 'vgg11', 'vgg19']
PRUNERS    = ['magnitude', 'snip', 'wanda']      # headline criterion first
SPARSITIES = (0.95, 0.97, 0.99, 0.999)
GPU        = 0

plan = []
for seed in SEEDS:
    # Dense first: every sparse cell of a seed resolves its checkpoint from the
    # dense run of the SAME seed, so the pairing holds all the way to init.
    plan += [nb.make_cell(m, 'dense', seed=seed) for m in MODELS]
    for p in PRUNERS:
        for s in SPARSITIES:
            for m in MODELS:
                plan.append(nb.make_cell(m, 'prune', seed=seed,
                                         pruner=p, sparsity=s))
                plan.append(nb.make_cell(m, 'bacp', seed=seed,
                                         pruner=p, sparsity=s))

n_dense = len(SEEDS) * len(MODELS)
n_pair  = len(SEEDS) * len(PRUNERS) * len(SPARSITIES) * len(MODELS)
print(f'{len(plan)} cells = {n_dense} dense + {n_pair} I.P. + {n_pair} BaCP')
assert len(plan) == n_dense + 2 * n_pair

# the first sparse pair must be the same cell in both arms, adjacent
a, b = plan[n_dense // len(SEEDS)], plan[n_dense // len(SEEDS) + 1]
assert a['key'].replace('.prune.', '.') == b['key'].replace('.bacp.', '.'), \
    f'arms are not interleaved: {a["key"]} then {b["key"]}'
print(f'interleaved, e.g.\n  {a["key"]}\n  {b["key"]}')
assert nb.sanity_check(plan), 'sanity check failed'

## Run

One row per epoch. `results.csv` is rewritten after every cell.

In [ ]:
for cell in plan:
    nb.run(cell, gpu=GPU)
    nb.update_results_csv()

## Progress --- re-run this cell any time

In [ ]:
import json, glob, os
root = os.environ['BACP_RESULTS_DIR']
seen = {}
for f in glob.glob(os.path.join(root, 'runs', '*.json')):
    r = json.load(open(f, encoding='utf-8'))
    k = r.get('experiment_group') or ''
    if r.get('status') == 'ok' and '.smoke' not in k and '.lr' not in k:
        seen[k] = r.get('test_acc_pct')

for seed in (1, *SEEDS):
    done = sum(1 for k in seen if k.endswith(f'.seed{seed}'))
    print(f'seed {seed}: {done:3d} / 100 cells')

print('\ncells with all three seeds:')
full = 0
for p in PRUNERS:
    for s in SPARSITIES:
        for m in MODELS:
            for arm in ('prune', 'bacp'):
                ks = [f'static.{arm}.{m}.cifar10.s{s}.{p}.seed{n}' for n in (1, *SEEDS)]
                if all(k in seen for k in ks):
                    full += 1
print(f'  {full} of {2*len(PRUNERS)*len(SPARSITIES)*len(MODELS)}')